# Module 3 — Code Along: OOP, Decorators, Type Hints, FastAPI (the bank-accounts story)

In Module 2 we **stored many account dicts** in files (JSON, CSV), split helpers into modules, and logged what happened. But an account is still a bare `dict` — nothing bundles its data with its behavior, and `deposit`/`withdraw` live in loose functions.

**Now we give an account real behavior by turning it into a class** — `class BankAccount` with `__init__`, methods, and dunder methods → then a `@dataclass` that writes the boilerplate for us, decorators (`@log_calls`, `@retry`) that wrap account operations, and a tiny FastAPI route that serves accounts.

Same canonical account shape, same owners (Ada, Lin, Sam) as M1/M2:

```python
id: int, owner: str, account_type: str, balance: float, is_active: bool, tags: list[str]
```

These are **minimal teaching demos** on the account domain — the real `Product`/`ProductCatalog` belongs to the lab, not here.

Run top to bottom. Cells are self-contained and re-runnable.

## class / self / __init__ — what is it?

A **class** is a blueprint; an **object** is one thing built from it. `__init__` runs once when you build the object and sets up that object's data.

In [ ]:
# A class is a blueprint. An object is one thing made from it.
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner          # self = THIS account
        self.balance = balance

    def get_owner(self):            # a read-only method (changes no state)
        return self.owner

    def deposit(self, amount):
        self.balance += amount
        return self.balance

acct = BankAccount("Ada", 100)      # __init__ runs once, builds the object
print(acct.owner, acct.balance)     # Ada 100


## self — what is it?

A method's first parameter is the object it was called on. Python passes it automatically — you write `self` by convention.

`acct.deposit(50)` becomes `BankAccount.deposit(acct, 50)` — Python fills in `self`.

In [ ]:
# Method-call sugar: `obj.method(args)` is `Class.method(obj, args)`.
# Show it on a NON-mutating read so the equivalence is clean (no state change).
acct = BankAccount("Ada", 100)

# `get_owner` just reads — both call forms are identical and print the same thing.
print(acct.get_owner())                  # Ada — `acct` became `self`
print(BankAccount.get_owner(acct))       # Ada — we pass `self` by hand

# `self` is just a name. The object is always the thing before the dot.

# A separate, mutating example — here state DOES change between calls (not 'identical'):
print(acct.deposit(50))                  # 150
print(acct.deposit(50))                  # 200 — balance kept growing


## methods vs attributes — what is it?

**Attributes** are the data on an object (`self.balance`). **Methods** are functions in the class that act on that data. Read an attribute; *call* a method.

In [ ]:
fresh = BankAccount("Lin", 100)

print(fresh.balance)        # attribute — a value you read, no parentheses
print(fresh.deposit)        # method object — not called yet
print(fresh.deposit(25))    # method CALL — runs it, returns 125
print(fresh.balance)        # 125 — the method changed the attribute

## dunder methods (__repr__, __eq__) — what is it?

"Dunder" = double underscore. Python calls these for you when you use built-in syntax. `__repr__` controls how an account prints; `__eq__` controls `==`. We compare accounts by **`id`** — two handles to the same account number are equal even if a balance differs.

In [ ]:
class BankAccount:
    def __init__(self, id, owner, balance=0.0):
        self.id = id
        self.owner = owner
        self.balance = balance

    def __repr__(self):                  # Python calls this when printing — a readable account summary
        return f"BankAccount(id={self.id}, owner={self.owner!r}, balance={self.balance})"

    def __eq__(self, other):             # Python calls this for ==
        if not isinstance(other, BankAccount):
            return NotImplemented        # not an account -> let Python fall back -> False, no crash
        return self.id == other.id       # identity is the account id, not the balance

ada = BankAccount(1, "Ada", 1500.0)
print(ada)                               # BankAccount(id=1, owner='Ada', balance=1500.0)  -> __repr__

# Same id = same account, even though the balance differs (one reloaded after a deposit).
print(ada == BankAccount(1, "Ada", 1600.0))   # True   -> __eq__ compares by id
print(ada == BankAccount(2, "Lin", 1500.0))    # False  -> different id
print(ada == "Ada")                            # False — guard returns NotImplemented, no crash


## type hints — what is it?

Annotations that say what type a value *should* be. Python does **not** enforce them at runtime — they exist for readers, editors, and tools (mypy, FastAPI, Pydantic). We must list type hints **now**, because the next concept — `@dataclass` — turns these very annotations into the class's fields.

In [ ]:
# Hints document the shape: a list of account dicts in, a float out.
def total_balance(accounts: list[dict]) -> float:
    return sum(a["balance"] for a in accounts)

accounts = [
    {"id": 1, "owner": "Ada", "balance": 1500.0},
    {"id": 3, "owner": "Sam", "balance": 40.0},
]
print(total_balance(accounts))   # 1540.0

# Not enforced: the hint says the param is a str, but Python runs it with anything (the gotcha).
def label(owner: str) -> str:
    return f"account holder: {owner}"

print(label("Ada"))   # account holder: Ada
print(label(123))     # account holder: 123  -> passed an int; Python did NOT stop us
# Editors and mypy WOULD flag that call. Hints are documentation that tools can check, not runtime checks.


## @dataclass — what is it?

A decorator that writes the boring class boilerplate for you. List the **canonical six fields** with type hints (the very annotations we just learned); you get `__init__`, `__repr__`, and `__eq__` for free. `tags` needs `field(default_factory=list)` so each account gets its **own** empty list, not one shared list.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class BankAccount:
    id: int
    owner: str
    account_type: str = "savings"
    balance: float = 0.0
    is_active: bool = True
    tags: list[str] = field(default_factory=list)   # own empty list per account, not shared

ada = BankAccount(1, "Ada", "savings", 1500.0, True, ["primary", "online"])
print(ada)                                  # BankAccount(id=1, owner='Ada', ...)  -> free __repr__
print(ada == BankAccount(1, "Ada", "savings", 1500.0, True, ["primary", "online"]))  # True -> free __eq__

# Defaults let a fresh account skip the optional fields:
lin = BankAccount(2, "Lin")
print(lin)                                  # tags is its OWN empty list, account_type defaults to 'savings'
# We wrote zero __init__ / __repr__ / __eq__ — the decorator generated them from the type hints.


## what a decorator is — build one from scratch

A decorator is a function that **takes a function and returns a function**. `@log_calls` above `withdraw` is exactly `withdraw = log_calls(withdraw)` — it wraps the account operation so every call is logged. Build it before using `@retry`.

In [ ]:
import functools

def log_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"-> calling {func.__name__}{args}")
        result = func(*args, **kwargs)
        print(f"<- {func.__name__} returned {result!r}")
        return result
    return wrapper

@log_calls                  # same as: withdraw = log_calls(withdraw)
def withdraw(acct, amount):
    acct["balance"] -= amount
    return acct["balance"]

ada = {"id": 1, "owner": "Ada", "balance": 1500.0}
print(withdraw(ada, 500))   # logs the call and the return, then prints 1000.0


## @retry(times=3) — a parametrized decorator

A parametrized decorator: you *call* it with arguments, then it decorates. One extra layer — the outer function captures the settings and returns the actual decorator. We wrap a flaky `fetch_account` (think: a network blip reading the account) that fails twice, then succeeds. The failure counter resets at the top of the cell, so re-running is reproducible.

In [ ]:
import functools

def retry(times=3):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last = None
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as exc:
                    last = exc
                    print(f"attempt {attempt} failed: {exc}")
            raise last
        return wrapper
    return decorator

# Module-level counter, RESET here so the notebook is reproducible (no randomness).
attempts = {"n": 0}

@retry(times=3)
def fetch_account(acct_id):
    attempts["n"] += 1
    if attempts["n"] < 3:                       # deterministic: fail on attempts 1 and 2
        raise ConnectionError("account service unavailable")
    return {"id": acct_id, "owner": "Ada", "balance": 1500.0}

print(fetch_account(1))   # two failures logged, then the account dict on attempt 3


## FastAPI is decorators in action — what is it?

A web route is just a function with a decorator on top. `@app.get("/accounts")` *registers* that function to run when a request hits `/accounts` — the same register-a-function shape you just built. The toy `MiniApp` below mimics it so the cell runs without a server; the real FastAPI is shown in a guarded `try/except`.

In [ ]:
# FastAPI's @app.get is a decorator that REGISTERS a function as a route.
# We don't need a running server to see the pattern — here's a toy that mimics it.
ACCOUNTS = [
    {"id": 1, "owner": "Ada", "balance": 1500.0},
    {"id": 2, "owner": "Lin", "balance": 0.0},
    {"id": 3, "owner": "Sam", "balance": 40.0},
]

class MiniApp:
    def __init__(self):
        self.routes = {}

    def get(self, path):                 # returns a decorator (parametrized!)
        def register(func):
            self.routes[path] = func     # store the function under its path
            return func
        return register

app = MiniApp()

@app.get("/accounts")                    # same shape as FastAPI's @app.get
def list_accounts():
    return ACCOUNTS

# A "request" is just looking up the path and calling the function.
print(app.routes["/accounts"]())         # the list of account dicts

# The real thing (FastAPI does exactly this, plus HTTP + auto /docs):
try:
    from fastapi import FastAPI
    real = FastAPI()

    @real.get("/accounts")
    def real_list_accounts():
        return ACCOUNTS

    print("fastapi installed — route registered:", [r.path for r in real.routes if getattr(r, "path", "") == "/accounts"])
except ImportError:
    print("fastapi not installed — toy MiniApp above shows the identical decorator pattern")


## That's Day 1 — where the account story goes next

Across three modules, one account grew up:

- **M1** — one account as a `dict`: variables, control flow, functions, exceptions.
- **M2** — many accounts in files: containers, comprehensions, JSON/CSV, modules, logging.
- **M3** — an account with real behavior: `class BankAccount` → `@dataclass`, decorators (`@log_calls`, `@retry`), and a FastAPI route serving accounts.

**Next, Day 2** migrates the `@dataclass BankAccount` to a **Pydantic `BaseModel`** — the same six fields, the same type hints, but now with **runtime validation** (a negative balance or a typo'd field is rejected at construction, not discovered later). The lab takes this account-shaped pattern and applies it to a `Product` / `ProductCatalog` with a full FastAPI CRUD server.